In [ ]:
!pip install geemap
import ee
import geemap.foliumap as geemap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 28.9 MB/s eta 0:00:00


In [ ]:
import ee
# Trigger authentication
ee.Authenticate()
# Initialize the library
ee.Initialize(project='project-487420')

In [ ]:
# Load administrative boundaries
countries = ee.FeatureCollection("FAO/GAUL/2015/level2")

# Filter Tizi Ouzou, Algeria
roi = countries.filter(
    ee.Filter.And(
        ee.Filter.eq('ADM0_NAME', 'Algeria'),
        ee.Filter.eq('ADM1_NAME', 'Tizi Ouzou')
    )
)

In [ ]:
collection = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
    .filterBounds(roi) \
    .filterDate("2021-08-20", "2021-09-10") \
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 30))

# Mosaic / Median
image = collection.median().clip(roi)

#  Compute spectral indices
# Burn index
nbr = image.normalizedDifference(['B8', 'B12']).rename('NBR')

# Vegetation index
ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')

# Water index
ndwi = image.normalizedDifference(['B3', 'B8']).rename('NDWI')

# Modified Soil Adjusted Vegetation Index
msavi = image.expression(
    "((2 * NIR + 1) - sqrt((2 * NIR + 1)**2 - 8*(NIR - RED)))/2",
    {'NIR': image.select('B8'), 'RED': image.select('B4')}
).rename('MSAVI')

# SWIR/NIR ratio
swir_nir = image.select('B12').divide(image.select('B8')).rename('SWIR_NIR')

#  Add all indices as bands
final_image = image.addBands([nbr, ndvi, ndwi, msavi, swir_nir])

# Add bands
final_image = image.addBands([nbr, ndvi, ndwi, msavi, swir_nir])

In [ ]:
#  Create interactive map
Map = geemap.Map(center=[36.7, 3.0], zoom=9)

#  Add True Color layer
Map.addLayer(final_image, {'bands':['B4','B3','B2'], 'max':3000}, 'True Color')

# Add NBR layer
Map.addLayer(nbr, {'min':-0.5, 'max':0.5, 'palette':['red','white','green']}, 'NBR')

#  Add NDVI layer
Map.addLayer(ndvi, {'min':-0.5, 'max':0.8, 'palette':['blue','white','green']}, 'NDVI')

#  Add NDWI layer
Map.addLayer(ndwi, {'min':-1, 'max':1, 'palette':['brown','white','blue']}, 'NDWI')

#  Add MSAVI layer
Map.addLayer(msavi, {'min':-0.5, 'max':0.8, 'palette':['brown','white','green']}, 'MSAVI')

#  Add SWIR/NIR ratio layer
Map.addLayer(swir_nir, {'min':0, 'max':2, 'palette':['yellow','white','purple']}, 'SWIR/NIR')

#  Add ROI boundary
Map.addLayer(roi, {'color':'black'}, 'Tizi Ouzou Boundary')



In [ ]:
#  Add Title
title_html = """
<div style="position: fixed; top: 10px; left: 50%; transform: translateX(-50%);
            z-index: 9999; background-color: white; padding: 10px;
            border: 2px solid black; border-radius: 5px;">
    <h3 style="margin: 0; font-size: 20px;"><b>Wildfire Analysis: Tizi Ouzou (August 2021)</b></h3>
</div>
"""
Map.add_html(title_html)

# NBR Legend
nbr_keys = ['Burned Area (Severe)', 'Bare Soil / Ash', 'Healthy Vegetation']
nbr_colors = ['#FF0000', '#FFFFFF', '#008000']
Map.add_legend(title='NBR: Burn Severity', keys=nbr_keys, colors=nbr_colors, position='bottomleft')

#  NDVI Legend
ndvi_keys = ['Water / No Veg', 'Sparse Veg', 'Dense Forest']
ndvi_colors = ['#0000FF', '#FFFFFF', '#008000']
Map.add_legend(title='NDVI: Vegetation', keys=ndvi_keys, colors=ndvi_colors, position='bottomright')

#  NDWI Legend
ndwi_keys = ['Soil / Low Moisture', 'Moderate', 'Water / High Moisture']
ndwi_colors = ['#A0522D', '#FFFFFF', '#0000FF']  # brown → white → blue
Map.add_legend(title='NDWI: Water Index', keys=ndwi_keys, colors=ndwi_colors, position='topright')

#  MSAVI Legend
msavi_keys = ['Bare Soil', 'Sparse Veg', 'Dense Vegetation']
msavi_colors = ['#A0522D', '#FFFFFF', '#008000']
Map.add_legend(title='MSAVI: Vegetation (Soil Adjusted)', keys=msavi_keys, colors=msavi_colors, position='topleft')
# SWIR/NIR Legend (use a valid position)
swir_nir_keys = ['Low Ratio', 'Medium', 'High Ratio']
swir_nir_colors = ['#FFFF00', '#FFFFFF', '#800080']
Map.add_legend(title='SWIR/NIR Ratio', keys=swir_nir_keys, colors=swir_nir_colors, position='topleft')
#  Layer Control
Map.add_layer_control()

Map

In [ ]:
# Generate training points
# Labels based on NBR
burned_mask = nbr.lt(-0.1)
unburned_mask = nbr.gt(0.3)

burned_samples = burned_mask.selfMask().stratifiedSample(
    numPoints=100, region=roi, scale=20, geometries=True
).map(lambda f: f.set('class', 1))

unburned_samples = unburned_mask.selfMask().stratifiedSample(
    numPoints=100, region=roi, scale=20, geometries=True
).map(lambda f: f.set('class', 0))

training_data = burned_samples.merge(unburned_samples)

In [ ]:
# Split into training/validation
with_random = training_data.randomColumn('random')
train_partition = with_random.filter(ee.Filter.lt('random', 0.7))
test_partition = with_random.filter(ee.Filter.gte('random', 0.7))


In [ ]:
 #Train Random Forest
feature_bands = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12', 'NBR', 'NDVI', 'NDWI', 'MSAVI', 'SWIR_NIR']

training = final_image.select(feature_bands).sampleRegions(
    collection=train_partition,
    properties=['class'],
    scale=20
)

classifier = ee.Classifier.smileRandomForest(100).train(
    features=training,
    classProperty='class',
    inputProperties=feature_bands
)


In [ ]:
#7️⃣ Validate on test set
# -----------------------------
test_data = final_image.select(feature_bands).sampleRegions(
    collection=test_partition,
    properties=['class'],
    scale=20
)

validated = test_data.classify(classifier)
confusion_matrix = validated.errorMatrix('class', 'classification')

print('Validation Confusion Matrix:', confusion_matrix.getInfo())
print('Validation Accuracy:', confusion_matrix.accuracy().getInfo())

In [ ]:
#  Classify the entire ROI

classified = final_image.select(feature_bands).classify(classifier)


#  Create interactive map
Map = geemap.Map(center=[36.7, 3.0], zoom=9)

Map.addLayer(classified, {'min': 0, 'max': 1, 'palette': ['green', 'red']}, 'Final Burn Classification')
Map.addLayer(roi, {'color': 'black'}, 'Tizi Ouzou Boundary')

# Title
title_html = """
<div style="position: fixed; top: 10px; left: 50%; transform: translateX(-50%);
            z-index: 9999; background-color: white; padding: 10px;
            border: 2px solid black; border-radius: 5px;">
    <h3 style="margin: 0; font-size: 20px;"><b>  – Tizi Ouzou, August 2021</b></h3>
</div>
"""
Map.add_html(title_html)

# Legends
Map.add_legend(title='NBR: Burn Severity', keys=['Burned', 'Unburned'], colors=['#FF0000', '#008000'], position='bottomleft')

Map.add_layer_control()
Map